# 油价对CPI与PPI的影响分析：直接效应与间接效应

## 研究概述
- **研究对象**：中国、美国、日本、越南
- **指标**：CPI（居民消费价格指数）、PPI（生产者价格指数）
- **效应分类**：直接效应（能源直接传导）、间接效应（生产链传导）
- **数据期**：2023年及之后
- **方法论**：基于文献弹性系数及数据回归估计

## 参考文献
1. Bachmeier, L., & Li, Q. (2007). *Pass-through of oil prices to domestic prices.* Energy Economics.
2. Chen, S. S. (2009). *Oil price pass-through into inflation.* Energy Economics, 31(1), 126-133.
3. Hamilton, J. D. (2003). *What is an oil shock?* Journal of Econometrics, 113(2), 363-398.
4. Cologni, A., & Manera, M. (2008). *Oil prices, inflation and interest rates in a structural cointegrated VAR model for the G-7 countries.* Energy Economics.

## 1. 加载数据

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
cwd = os.getcwd()
cpi_dir = os.path.join(cwd, 'cpi_ppi_analysis') if os.path.exists(os.path.join(cwd, 'cpi_ppi_analysis')) else cwd
if cpi_dir != cwd:
    os.chdir(cpi_dir)
sys.path.insert(0, cpi_dir)

from data_loader import load_fred_data, load_sample_data
from effects_calculator import (
    compute_direct_effect,
    compute_indirect_effect,
    compute_all_effects,
    compute_effects_from_series,
    estimate_elasticity_from_data,
)

# 优先尝试FRED，若无API key则使用样本数据
START_DATE = "2023-01-01"
df = load_fred_data(start_date=START_DATE)

if df.empty or len(df) < 6:
    print("使用样本数据 (2023-2024)")
    df = pd.read_csv("data/sample_data_2023.csv", parse_dates=["date"], index_col="date")
else:
    print("已从FRED加载数据")
    if "us_cpi" in df.columns and "us_ppi" not in df.columns:
        # 若只有部分系列，补充其他国家的样本数据
        sample = pd.read_csv("data/sample_data_2023.csv", parse_dates=["date"], index_col="date")
        df = df.join(sample[[c for c in sample.columns if c not in df.columns]], how="outer")

df = df[df.index >= START_DATE].dropna(how="all")
print(f"数据期: {df.index.min()} 至 {df.index.max()}")
df.tail()

## 2. 油价变动与描述统计

In [ ]:
# 确定油价列名
oil_col = "oil_wti" if "oil_wti" in df.columns else "oil" if "oil" in df.columns else df.columns[0]
oil = df[oil_col].dropna()

# 月度同比变动
oil_pct = oil.pct_change() * 100
print("=== 2023年以来油价月度变动 (%) ===")
print(oil_pct.describe())

# 基准期到当前的总变动
if len(oil) >= 2:
    total_pct = (oil.iloc[-1] / oil.iloc[0] - 1) * 100
    print(f"\n样本期油价总变动: {total_pct:.2f}%")
    print(f"基准 (2023-01): {oil.iloc[0]:.2f} 美元/桶")
    print(f"期末: {oil.iloc[-1]:.2f} 美元/桶")

## 3. 直接效应与间接效应计算（基于文献弹性）

### 情景分析：油价上涨 10%

In [ ]:
results_10pct = compute_all_effects(oil_pct_change=10)
print("=== 油价上涨10%时的直接效应与间接效应 ===")
print(results_10pct.to_string(index=False))

results_10pct

### 情景分析：油价上涨 20%

In [ ]:
results_20pct = compute_all_effects(oil_pct_change=20)
print("=== 油价上涨20%时的直接效应与间接效应 ===")
print(results_20pct.to_string(index=False))

results_20pct

### 基于样本期实际油价变动的效应估算

In [ ]:
# 构建各国价格字典（列名可能不同）
price_dict = {}

country_col_map = {
    "USA": ["us_cpi", "usa_cpi", "US_CPI"],
    "China": ["china_cpi", "chn_cpi"],
    "Japan": ["japan_cpi", "jpn_cpi"],
    "Vietnam": ["vietnam_cpi", "vnm_cpi"],
}
ppi_map = {
    "USA": ["us_ppi", "usa_ppi"],
    "China": ["china_ppi", "chn_ppi"],
    "Japan": ["japan_ppi", "jpn_ppi"],
    "Vietnam": ["vietnam_ppi", "vnm_ppi"],
}

for country, cpi_cols in country_col_map.items():
    cpi_series = None
    for c in cpi_cols:
        if c in df.columns:
            cpi_series = df[c].dropna()
            break
    ppi_cols = ppi_map.get(country, [])
    ppi_series = None
    for c in ppi_cols:
        if c in df.columns:
            ppi_series = df[c].dropna()
            break
    if cpi_series is not None or ppi_series is not None:
        price_dict[country] = {}
        if cpi_series is not None:
            price_dict[country]["CPI"] = cpi_series
        if ppi_series is not None:
            price_dict[country]["PPI"] = ppi_series

if price_dict and oil_col in df.columns:
    results_actual = compute_effects_from_series(
        df[oil_col], price_dict, base_date=START_DATE
    )
    print("=== 基于样本期实际油价平均变动的效应 ===")
    print(results_actual.to_string(index=False))
else:
    print("无法构建完整价格序列，请检查数据列名")

## 4. 数据驱动的弹性估计（美国示例）

若数据充足，可从数据估计弹性以验证/补充文献系数。

In [ ]:
cpi_col = "us_cpi" if "us_cpi" in df.columns else "usa_cpi"
ppi_col = "us_ppi" if "us_ppi" in df.columns else "usa_ppi"

if cpi_col in df.columns and oil_col in df.columns:
    d_elast, i_elast = estimate_elasticity_from_data(df[oil_col], df[cpi_col], lag=2)
    print(f"美国CPI - 数据估计弹性: 直接={d_elast:.4f}, 间接={i_elast:.4f}")
if ppi_col in df.columns and oil_col in df.columns:
    d_elast, i_elast = estimate_elasticity_from_data(df[oil_col], df[ppi_col], lag=2)
    print(f"美国PPI - 数据估计弹性: 直接={d_elast:.4f}, 间接={i_elast:.4f}")

print("\n注：样本期较短时估计可能不稳定，文献系数更稳健。")

## 5. 汇总表：各国CPI/PPI对油价变动的敏感性

In [ ]:
import pandas as pd

# 多情景汇总
scenarios = [5, 10, 15, 20, 30]
all_res = []
for s in scenarios:
    r = compute_all_effects(s)
    r["情景"] = f"油价+{s}%"
    all_res.append(r)

summary = pd.concat(all_res, ignore_index=True)
pivot_direct = summary.pivot_table(
    index=["国家", "指标"], columns="情景", values="直接效应(%)"
)
pivot_indirect = summary.pivot_table(
    index=["国家", "指标"], columns="情景", values="间接效应(%)"
)

print("=== 直接效应 (%) ===")
print(pivot_direct)
print("\n=== 间接效应 (%) ===")
print(pivot_indirect)

## 6. 可视化

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 直接效应对比
ax1 = axes[0, 0]
r10 = compute_all_effects(10)
direct = r10[r10["指标"]=="CPI"][["国家", "直接效应(%)"]].set_index("国家")
direct.plot(kind="bar", ax=ax1, legend=False, color="steelblue")
ax1.set_title("CPI直接效应 (油价+10%)")
ax1.set_ylabel("百分点")

ax2 = axes[0, 1]
indirect = r10[r10["指标"]=="CPI"][["国家", "间接效应(%)"]].set_index("国家")
indirect.plot(kind="bar", ax=ax2, legend=False, color="coral")
ax2.set_title("CPI间接效应 (油价+10%)")
ax2.set_ylabel("百分点")

ax3 = axes[1, 0]
direct_ppi = r10[r10["指标"]=="PPI"][["国家", "直接效应(%)"]].set_index("国家")
direct_ppi.plot(kind="bar", ax=ax3, legend=False, color="steelblue")
ax3.set_title("PPI直接效应 (油价+10%)")
ax3.set_ylabel("百分点")

ax4 = axes[1, 1]
indirect_ppi = r10[r10["指标"]=="PPI"][["国家", "间接效应(%)"]].set_index("国家")
indirect_ppi.plot(kind="bar", ax=ax4, legend=False, color="coral")
ax4.set_title("PPI间接效应 (油价+10%)")
ax4.set_ylabel("百分点")

plt.tight_layout()
plt.savefig("cpi_ppi_effects_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("图表已保存为 cpi_ppi_effects_summary.png")

## 7. 结论与参考文献

### 主要结论
- **直接效应**：油价上升主要经由能源成分直接推高CPI/PPI，日本、美国对能源价格更敏感。
- **间接效应**：通过生产链传导，越南、中国等制造业国间接效应相对显著。
- **PPI**：相比CPI，PPI对油价的直接与间接弹性均更高，符合“成本→出厂价→零售价”的传导逻辑。

### 参考文献
1. Bachmeier, L., & Li, Q. (2007). Pass-through of oil prices to domestic prices: Evidence from an oil-importing and an oil-exporting country. *Energy Economics*.
2. Chen, S. S. (2009). Oil price pass-through into inflation. *Energy Economics*, 31(1), 126-133.
3. Hamilton, J. D. (2003). What is an oil shock? *Journal of Econometrics*, 113(2), 363-398.
4. Cologni, A., & Manera, M. (2008). Oil prices, inflation and interest rates in a structural cointegrated VAR model for the G-7 countries. *Energy Economics*, 30(3), 856-888.
5. Hooker, M. A. (2002). Are oil shocks inflationary? Asymmetric and nonlinear specifications versus changes in regime. *Journal of Money, Credit, and Banking*.
6. Nakajima, T. (2011). Monetary policy transmission under zero interest rates. *IMF Economic Review*.